# BTS Digital Twin (NVS) — Đóng gói submission từ checkpoint có sẵn

Notebook này **KHÔNG train** — chỉ tải 8 checkpoint (`gs_model/`) của 8 scene
`private_set1` đã train ở `kaggle_private.ipynb` (đã upload lên Google Drive), render
lại ảnh tại các pose test rồi đóng gói `submission_round1.zip`. Chạy nhanh (không tốn
GPU-giờ cho train, chỉ tốn cho render — vài phút/scene).

**Trước khi chạy, cần điền:**
1. Settings → Accelerator: **GPU T4 x2** (hoặc P100) → Internet: **On**.
2. `REPO_URL` ở Bước 3, `GDRIVE_URL` ở Bước 4 (dataset — cần `test_poses.csv` từng scene).
3. `CHECKPOINT_LINKS` ở Bước 6 — dán đủ 8 link Google Drive (mỗi scene 1 link **thư mục**
   `gs_model` đã upload theo hướng dẫn ở Bước 7 của `kaggle_private.ipynb`).

**Bảo mật:** để notebook này **Private**.

## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)

In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Bước build
2 CUDA extension (`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Nếu push CẢ project (gồm `Đề bài.md`, `KE_HOACH_VONG1.md`, `Dataset/`, `pipeline/`...)
làm 1 repo cũng được — cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/`
và `scripts/`) ở bất kỳ độ sâu nào trong repo, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin.git"
REPO_BRANCH = "feature/antenna-region-focus"  # <-- QUAN TRỌNG: đổi thành "main" nếu không cần tính năng antenna-focus (branch này mới có 07_build_antenna_weights.py/apply_antenna_patch.py — clone nhầm "main" thì Bước antenna-focus sẽ báo lỗi thiếu file)

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 --branch "{REPO_BRANCH}" "{clone_url}" /kaggle/working/_repo_clone
print(f"Đã clone branch: {REPO_BRANCH}")

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA/phase1/{public_set,private_set1}/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA` cũng
được — cell dưới tự dò tìm thư mục `phase1` ở bất kỳ độ sâu nào trong zip).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/1GUflBBz4hrVtkMcHLfLMwUYk4HsJkXFu/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục phase1 ...")

In [ ]:
# Tự dò thư mục "phase1" (chứa public_set/ hoặc private_set1/) ở bất kỳ đâu trong
# zip vừa giải nén, rồi symlink về đúng vị trí mà pipeline/common/scenes.py cần:
#   /kaggle/working/Dataset/VAI_NVS_DATA/phase1
import os
from pathlib import Path

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (public_set/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/phase1" trước "phase1" thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    p = Path(dirpath)
    if p.name == "phase1" and (("public_set" in dirnames) or ("private_set1" in dirnames)):
        found = p
        break

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'phase1' chứa public_set/private_set1 trong zip vừa giải nén.\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — kiểm tra lại cấu trúc zip bạn đã upload lên Google Drive."
    )

print("Tìm thấy:", found)
target_parent = Path("/kaggle/working/Dataset/VAI_NVS_DATA")
target_parent.mkdir(parents=True, exist_ok=True)
target = target_parent / "phase1"
if target.exists() or target.is_symlink():
    target.unlink() if target.is_symlink() else None
if not target.exists():
    os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 13 scene + scene nào có sparse hợp lệ — dataset đầy đủ
# thì kỳ vọng has_valid_provided_sparse=True cho CẢ 13 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại giá trị BTS_DATASET_ROOT ở cell
# trên có trỏ đúng chỗ chứa public_set/private_set1 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá /kaggle/working/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.split:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 — Tải checkpoint từ Google Drive cho cả 8 scene

Điền đủ 8 link Google Drive (mỗi scene 1 link **thư mục** `gs_model` — thư mục thô,
không phải file zip — đã upload theo hướng dẫn ở Bước 7 của `kaggle_private.ipynb`)
vào dict `CHECKPOINT_LINKS` bên dưới. Nhớ share thư mục đó ở chế độ "Anyone with the link".

In [ ]:
CHECKPOINT_LINKS = {
    "HCM0249": "",  # <-- dán link Google Drive (thư mục gs_model) của scene này
    "HCM0254": "",
    "HCM0276": "",
    "HCM1439": "",
    "HNI0131": "",
    "HNI0265": "",
    "HNI0366": "",
    "HNI0437": "",
}

missing = [s for s, link in CHECKPOINT_LINKS.items() if not link]
assert not missing, f"Chưa điền link Google Drive cho scene: {missing}"

In [ ]:
import shutil
from pathlib import Path

for scene, link in CHECKPOINT_LINKS.items():
    dest_dir = Path(f"/kaggle/working/pipeline/work/{scene}")
    dest_dir.mkdir(parents=True, exist_ok=True)
    tmp_dir = Path(f"/kaggle/working/_ckpt_dl_{scene}")
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"===== {scene}: tải checkpoint (thư mục Google Drive) =====")
    !gdown --folder "{link}" -O "{tmp_dir}"

    # Tự dò thư mục con tên "point_cloud" ở bất kỳ độ sâu nào trong dữ liệu vừa tải
    # (không ép tên thư mục Drive phải đúng là "gs_model" — tuỳ bạn đặt tên lúc upload)
    # rồi coi thư mục CHA của nó chính là gs_model, move về đúng chỗ pipeline cần.
    matches = list(tmp_dir.rglob("point_cloud"))
    assert matches, (f"{scene}: không tìm thấy thư mục 'point_cloud' trong dữ liệu tải "
                     f"về ({tmp_dir}) — kiểm tra lại link Drive có đúng thư mục gs_model không.")
    gs_model_dst = dest_dir / "gs_model"
    shutil.rmtree(gs_model_dst, ignore_errors=True)
    shutil.move(str(matches[0].parent), str(gs_model_dst))
    shutil.rmtree(tmp_dir, ignore_errors=True)

    ply_glob = list((gs_model_dst / "point_cloud").glob("iteration_*/point_cloud.ply"))
    assert ply_glob, f"{scene}: không thấy point_cloud.ply trong {gs_model_dst} — kiểm tra lại dữ liệu Drive."
    print(f"-> OK, {len(ply_glob)} checkpoint iteration cho {scene}")

## Bước 6 — Render lại ảnh test cho cả 8 scene

Dùng đúng checkpoint vừa tải (iteration lớn nhất có sẵn cho mỗi scene, mặc định của
`04_render_test_poses.py`).

In [ ]:
for scene in CHECKPOINT_LINKS:
    !python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {scene}

## Bước 7 — Đóng gói + kiểm tra `submission_round1.zip`

Script tự kiểm tra đủ 8 scene / đủ ảnh / đúng kích thước trước khi nén, và verify
lại chính file zip vừa tạo (xem `KE_HOACH_VONG1.md` mục 7 — checklist trước khi nộp).

In [ ]:
!python /kaggle/working/pipeline/scripts/06_package_submission.py \
    --out /kaggle/working/submission_round1.zip \
    --filename_mode literal

## Bước 8 — Lưu kết quả

`submission_round1.zip` đã nằm ở `/kaggle/working/` — bấm **Save Version** (góc
trên phải) để Kaggle giữ lại file này trong tab "Output" của notebook, tải về từ đó
để nộp bài.